# LLM Turing Test Results

Summary of the three runs across splits: `train_1`, `train_2`, and `test`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.grid"] = False
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"

RUNS = {
    "train_1": Path("results/runs/20260515_235653_train_1"),
    "train_2": Path("results/runs/20260515_235720_train_2"),
    "test": Path("results/runs/20260515_235747_test"),
}

JUDGE_FILES = ["turing_results_claude-opus-4.6.json", "turing_results_gpt-5.4.json"]

EXPORT_DIR = Path("../report/data")


In [ ]:
def load_result(path):
    with open(path) as f:
        data = json.load(f)
    if "error" in data:
        return None
    return data

rows = []
raw_data = {}
for split, run_dir in RUNS.items():
    for jf in JUDGE_FILES:
        p = run_dir / jf
        if not p.exists():
            continue
        data = load_result(p)
        if data is None:
            continue
        judge = data["judge_name"]
        a = data["absolute"]
        cu = data.get("comparative_unsure", {})
        raw_data[(split, judge)] = data
        rows.append({
            "split": split,
            "judge": judge,
            "n_pairs": data["n_pairs"],
            "abs_accuracy": a["accuracy"],
            "abs_acc_sim": a["accuracy_sim"],
            "abs_acc_real": a["accuracy_real"],
            "likert_sim": a["avg_likert_sim"],
            "likert_real": a["avg_likert_real"],
            "confusion": a["confusion"],
            "comp_acc": cu.get("accuracy_decided"),
            "comp_unsure_pct": cu["unsure"] / cu["total"] if cu.get("total") else None,
        })

df = pd.DataFrame(rows)
print(f"Loaded {len(df)} results across {df['split'].nunique()} splits and {df['judge'].nunique()} judges")
df[["split", "judge", "n_pairs", "abs_accuracy", "abs_acc_sim", "abs_acc_real", "likert_sim", "likert_real", "comp_acc"]]

## Summary Table

In [ ]:
def fmt(v, spec=".0%"):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "-"
    return f"{v:{spec}}"

table_rows = []
for _, r in df.iterrows():
    c = r["confusion"]
    table_rows.append({
        "Split": r["split"],
        "Judge": r["judge"],
        "N pairs": r["n_pairs"],
        "Abs Acc": fmt(r["abs_accuracy"]),
        "Acc(sim)": fmt(r["abs_acc_sim"]),
        "Acc(real)": fmt(r["abs_acc_real"]),
        "Likert(sim)": fmt(r["likert_sim"], ".2f"),
        "Likert(real)": fmt(r["likert_real"], ".2f"),
        "Comp Acc": fmt(r["comp_acc"]),
        "TP": c["tp"], "TN": c["tn"], "FP": c["fp"], "FN": c["fn"],
    })

summary = pd.DataFrame(table_rows)
summary

## Confusion Matrices

In [ ]:
split_order = ["train_1", "train_2", "test"]
splits_present = [s for s in split_order if s in df["split"].values]
judges_ordered = sorted(df["judge"].unique())

n_rows = len(judges_ordered)
n_cols = len(splits_present)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(3.5 * n_cols, 3.5 * n_rows),
)
if n_rows == 1:
    axes = axes[np.newaxis, :]
if n_cols == 1:
    axes = axes[:, np.newaxis]

cmap = plt.cm.GnBu

for j_idx, judge in enumerate(judges_ordered):
    for s_idx, split in enumerate(splits_present):
        ax = axes[j_idx, s_idx]
        subset = df[(df["split"] == split) & (df["judge"] == judge)]

        if len(subset) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes, fontsize=10, color="grey")
            ax.set_xticks([0, 1])
            ax.set_yticks([0, 1])
            ax.set_xticklabels([])
            ax.set_yticklabels([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.tick_params(left=False, bottom=False)
        else:
            c = subset.iloc[0]["confusion"]
            cm = np.array([[c["tn"], c["fp"]],
                           [c["fn"], c["tp"]]])

            im = ax.imshow(cm, cmap=cmap, vmin=0, vmax=cm.max() or 1, aspect="equal")
            for i in range(2):
                for jj in range(2):
                    total = cm[i].sum()
                    pct = cm[i, jj] / total * 100 if total > 0 else 0
                    ax.text(jj, i, f"{cm[i, jj]}\n({pct:.0f}%)",
                            ha="center", va="center", fontsize=9,
                            color="white" if cm[i, jj] > cm.max() * 0.6 else "black")

            ax.set_xticks([0, 1])
            ax.set_yticks([0, 1])
            ax.set_yticklabels(["Real", "Simulated"], fontsize=9)
            ax.set_xticklabels(["Real", "Simulated"], fontsize=9)
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.tick_params(left=False, bottom=False)

        if j_idx == 0:
            n_pairs = int(subset.iloc[0]["n_pairs"]) if len(subset) else "?"
            ax.set_title(f"{split} (N={n_pairs})", fontsize=11)

        if s_idx == 0:
            ax.set_ylabel(judge, fontsize=10, labelpad=10)

fig.supxlabel("Classified", fontsize=11, y=0.02)
fig.supylabel("Ground Truth", fontsize=11, x=0.01)
fig.suptitle("Confusion Matrices", fontsize=13)
plt.tight_layout(rect=[0.05, 0.05, 1, 0.95])

plt.savefig(EXPORT_DIR / "llm_confusion_matrices.pdf", bbox_inches="tight", dpi=150)
plt.show()

## Percentage of Conversations Judged as Real

In [ ]:
SIM_COLOR = "#E74C3C"
REAL_COLOR = "#2E86C1"

split_order = ["train_1", "train_2", "test"]
splits_present = [s for s in split_order if s in df["split"].values]
judges_ordered = sorted(df["judge"].unique())

fig, axes = plt.subplots(1, len(judges_ordered),
                         figsize=(7 * len(judges_ordered), 5), sharey=True)
if len(judges_ordered) == 1:
    axes = [axes]

for ax, judge in zip(axes, judges_ordered):
    jdf = df[df["judge"] == judge].set_index("split").reindex(splits_present)
    jdf = jdf.dropna(subset=["abs_acc_sim"])
    labels = jdf.index.tolist()
    x = np.arange(len(labels))
    width = 0.25

    sim_judged_real = [1 - row["abs_acc_sim"] for _, row in jdf.iterrows()]
    real_judged_real = [row["abs_acc_real"] for _, row in jdf.iterrows()]
    ns = [int(row["n_pairs"]) for _, row in jdf.iterrows()]

    ax.bar(x - width / 2, sim_judged_real, width, color=SIM_COLOR)
    ax.bar(x + width / 2, real_judged_real, width, color=REAL_COLOR)

    for i, (sv, rv) in enumerate(zip(sim_judged_real, real_judged_real)):
        ax.text(x[i] - width / 2, sv + 0.01, f"{sv:.0%}", ha="center", fontsize=8)
        ax.text(x[i] + width / 2, rv + 0.01, f"{rv:.0%}", ha="center", fontsize=8)
        ax.text(x[i] - width / 2, -0.06, f"N={ns[i]}", ha="center", fontsize=7, color="grey")
        ax.text(x[i] + width / 2, -0.06, f"N={ns[i]}", ha="center", fontsize=7, color="grey")

    ax.axhline(0.5, color="grey", linestyle="--", alpha=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_title(f"{judge} as judge")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.set_ylim(-0.1, 1.15)

    ax.spines["top"].set_visible(True)
    ax.spines["right"].set_visible(True)
    ax.spines["bottom"].set_visible(True)
    ax.spines["left"].set_visible(True)
    for spine in ax.spines.values():
        spine.set_edgecolor("#AAAAAA")
        spine.set_linewidth(0.7)
    ax.tick_params(axis="both", width=0.7)

axes[0].set_ylabel("% judged as real (higher = better simulator)")

fig.legend(
    handles=[
        Patch(facecolor=SIM_COLOR, label="simulated conversations"),
        Patch(facecolor=REAL_COLOR, label="real conversations"),
        Line2D([0], [0], color="grey", linestyle="--", alpha=0.5, label="chance (random guessing)"),
    ],
    loc="lower center", ncol=3, frameon=True, fontsize=10,
    bbox_to_anchor=(0.5, -0.02),
)
fig.suptitle("Percentage of Conversations Judged as Real", fontsize=14)
plt.tight_layout(rect=[0, 0.06, 1, 1])

plt.savefig(EXPORT_DIR / "llm_pct_judged_real.pdf", bbox_inches="tight", dpi=150)
plt.show()

## Sequential Rater Analysis (Simulating Human Protocol)

Each LLM judge acts as 10 independent raters, each seeing 10 conversations (5 real + 5 simulated) sequentially — matching the [human Turing test protocol](../humans/human_turing_test_stats_protocol.md).

**Primary estimand**: simulation detection rate $\hat{p} = \text{TP} / (\text{TP} + \text{FN})$

**Hypothesis**: $H_0: p \geq 0.65$ vs $H_1: p < 0.65$ — the simulator passes if the one-sided 95% upper confidence bound is below 0.65.

**Why Wilson?** Confidence intervals use the Wilson score method, consistent with the human rater analysis and the main-text figures. For proportions near 0 or 1 (such as real-recognition rates close to 100%) the Wilson interval stays within [0, 1] and has better small-sample coverage than the normal-approximation (Wald) interval. The protocol specifies it as the default for the primary analysis (section 9).

In [ ]:
from scipy import stats

SEQUENTIAL_FILES = {
    "claude-opus-4.6": Path("results/runs/20260518_170302_sequential/sequential_claude-opus-4.6.json"),
    "gpt-5.4": Path("results/runs/20260601_165135_sequential/sequential_gpt-5.4.json"),
}

# Load all individual ratings
all_ratings = []
for judge_name, path in SEQUENTIAL_FILES.items():
    with open(path) as f:
        sessions = json.load(f)
    for session in sessions:
        for r in session["ratings"]:
            all_ratings.append({
                "judge": judge_name,
                "rater_id": r["rater_id"],
                "conversation_id": r["conversation_id"],
                "ground_truth": r["ground_truth"],
                "classification": r["classification"],
                "classified_simulated": r["classified_simulated"],
                "correct": r["correct"],
                "likert_score": r.get("likert_score"),
                "presentation_order": r["presentation_order"],
            })

seq_df = pd.DataFrame(all_ratings)
print(f"Loaded {len(seq_df)} ratings from {seq_df['rater_id'].nunique()} raters across {seq_df['judge'].nunique()} models")
print(f"  Simulated judgements: {(seq_df['ground_truth'] == 'simulated').sum()}")
print(f"  Real judgements: {(seq_df['ground_truth'] == 'real').sum()}")

In [ ]:
def wilson_ci(k, n, alpha=0.05):
    """Two-sided Wilson score interval for a proportion."""
    if n == 0:
        return np.nan, np.nan
    p = k / n
    z = stats.norm.ppf(1 - alpha / 2)
    denom = 1.0 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = (z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / denom
    return max(centre - half, 0.0), min(centre + half, 1.0)

def wilson_upper(k, n, alpha=0.05):
    """One-sided Wilson score upper bound for a proportion."""
    if n == 0:
        return np.nan
    p = k / n
    z = stats.norm.ppf(1 - alpha)
    denom = 1.0 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = (z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / denom
    return min(centre + half, 1.0)


def compute_protocol_metrics(subset_df, label):
    """Compute all metrics from the stats protocol for a subset of ratings."""
    sim = subset_df[subset_df["ground_truth"] == "simulated"]
    real = subset_df[subset_df["ground_truth"] == "real"]

    tp = sim["classified_simulated"].sum()
    fn = len(sim) - tp
    fp = real["classified_simulated"].sum()
    tn = len(real) - fp
    n_sim = len(sim)
    n_real = len(real)

    det_rate = tp / n_sim if n_sim else 0
    fooled_rate = fn / n_sim if n_sim else 0
    specificity = tn / n_real if n_real else 0
    false_accusation = fp / n_real if n_real else 0
    accuracy = (tp + tn) / (tp + fn + fp + tn) if (tp + fn + fp + tn) else 0
    balanced_acc = 0.5 * det_rate + 0.5 * specificity

    ci_lo, ci_hi = wilson_ci(tp, n_sim)
    upper_one_sided = wilson_upper(tp, n_sim)
    passes = upper_one_sided < 0.65

    return {
        "label": label,
        "n_raters": subset_df["rater_id"].nunique(),
        "n_sim": n_sim,
        "n_real": n_real,
        "tp": tp, "fn": fn, "fp": fp, "tn": tn,
        "det_rate": det_rate,
        "fooled_rate": fooled_rate,
        "specificity": specificity,
        "false_accusation": false_accusation,
        "accuracy": accuracy,
        "balanced_acc": balanced_acc,
        "ci_95_lo": ci_lo,
        "ci_95_hi": ci_hi,
        "upper_one_sided_95": upper_one_sided,
        "passes_H0": passes,
    }

# Per-model only
results_list = []
for judge_name in SEQUENTIAL_FILES:
    subset = seq_df[seq_df["judge"] == judge_name]
    results_list.append(compute_protocol_metrics(subset, judge_name))

seq_results = pd.DataFrame(results_list)

# Display primary results
display_cols = [
    "label", "n_raters", "n_sim",
    "det_rate", "ci_95_lo", "ci_95_hi", "upper_one_sided_95", "passes_H0",
    "fooled_rate", "specificity", "balanced_acc",
    "tp", "fn", "fp", "tn",
]
styled = seq_results[display_cols].copy()
for col in ["det_rate", "ci_95_lo", "ci_95_hi", "upper_one_sided_95", "fooled_rate", "specificity", "balanced_acc"]:
    styled[col] = styled[col].map(lambda x: f"{x:.1%}")
styled.columns = [
    "Analysis", "Raters", "N(sim)",
    "Det. Rate", "CI low", "CI high", "Upper 1-sided", "Passes (p<0.65)?",
    "Fooled Rate", "Specificity", "Balanced Acc",
    "TP", "FN", "FP", "TN",
]
styled

In [ ]:
# Per-rater breakdown
rater_rows = []
for _, row in seq_df.groupby("rater_id"):
    rater_id = row.iloc[0]["rater_id"]
    judge = row.iloc[0]["judge"]
    sim = row[row["ground_truth"] == "simulated"]
    real = row[row["ground_truth"] == "real"]
    tp = int(sim["classified_simulated"].sum())
    fn = len(sim) - tp
    fp = int(real["classified_simulated"].sum())
    tn = len(real) - fp
    rater_rows.append({
        "Judge": judge,
        "Rater": rater_id,
        "Det. Rate": f"{tp}/{len(sim)} = {tp/len(sim):.0%}" if len(sim) else "-",
        "Specificity": f"{tn}/{len(real)} = {tn/len(real):.0%}" if len(real) else "-",
        "TP": tp, "FN": fn, "FP": fp, "TN": tn,
    })

rater_df = pd.DataFrame(rater_rows)
rater_df

In [ ]:
# Sequential protocol confusion matrices as figure
judges_ordered_seq = sorted(SEQUENTIAL_FILES.keys())
fig, axes = plt.subplots(1, len(judges_ordered_seq), figsize=(5 * len(judges_ordered_seq), 4))
if len(judges_ordered_seq) == 1:
    axes = [axes]

cmap = plt.cm.GnBu
labels = ["Real", "Simulated"]

for ax, judge in zip(axes, judges_ordered_seq):
    subset = seq_df[seq_df["judge"] == judge]
    sim = subset[subset["ground_truth"] == "simulated"]
    real = subset[subset["ground_truth"] == "real"]
    tp = int(sim["classified_simulated"].sum())
    fn = len(sim) - tp
    fp = int(real["classified_simulated"].sum())
    tn = len(real) - fp
    cm = np.array([[tn, fp], [fn, tp]])

    im = ax.imshow(cm, cmap=cmap, vmin=0, vmax=cm.max() or 1, aspect="equal")
    for i in range(2):
        for j in range(2):
            total_row = cm[i].sum()
            pct = cm[i, j] / total_row * 100 if total_row else 0
            ax.text(j, i, f"{cm[i, j]}\n({pct:.0f}%)",
                    ha="center", va="center", fontsize=11,
                    color="white" if cm[i, j] > cm.max() * 0.6 else "black")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel("Classified", fontsize=10)
    ax.set_ylabel("Ground Truth", fontsize=10)
    ax.set_title(judge, fontsize=11)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(left=False, bottom=False)

fig.suptitle("Sequential Protocol: Confusion Matrices", fontsize=13)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "llm_sequential_confusion.pdf", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Per-rater bar plot: % judged as real, split by ground truth (sim vs real)
judges_ordered_seq = sorted(SEQUENTIAL_FILES.keys())

fig, axes = plt.subplots(len(judges_ordered_seq), 1,
                         figsize=(18, 4.5 * len(judges_ordered_seq)), sharex=True, sharey=True)
if len(judges_ordered_seq) == 1:
    axes = [axes]

for ax, judge in zip(axes, judges_ordered_seq):
    judge_df = seq_df[seq_df["judge"] == judge]
    raters = sorted(judge_df["rater_id"].unique(),
                    key=lambda r: int(r.rsplit("_", 1)[-1]))
    x = np.arange(len(raters))
    width = 0.3

    sim_judged_real = []
    real_judged_real = []
    for rater in raters:
        rater_data = judge_df[judge_df["rater_id"] == rater]
        sim_data = rater_data[rater_data["ground_truth"] == "simulated"]
        real_data = rater_data[rater_data["ground_truth"] == "real"]
        sim_judged_real.append((~sim_data["classified_simulated"]).sum() / len(sim_data) if len(sim_data) else 0)
        real_judged_real.append((~real_data["classified_simulated"]).sum() / len(real_data) if len(real_data) else 0)

    ax.bar(x - width / 2, sim_judged_real, width, color=SIM_COLOR)
    ax.bar(x + width / 2, real_judged_real, width, color=REAL_COLOR)

    for i, (sv, rv) in enumerate(zip(sim_judged_real, real_judged_real)):
        ax.text(x[i] - width / 2, sv + 0.02, f"{sv:.0%}", ha="center", fontsize=7)
        ax.text(x[i] + width / 2, rv + 0.02, f"{rv:.0%}", ha="center", fontsize=7)

    ax.axhline(0.5, color="grey", linestyle="--", alpha=0.5)
    short_labels = [str(int(r.rsplit("_", 1)[-1])) for r in raters]
    ax.set_xticks(x)
    ax.set_xticklabels(short_labels, fontsize=9)
    ax.set_ylabel("% judged as real")
    ax.set_title(f"{judge}", fontsize=11)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.set_ylim(-0.05, 1.15)

    for spine in ax.spines.values():
        spine.set_edgecolor("#AAAAAA")
        spine.set_linewidth(0.7)
    ax.tick_params(axis="both", width=0.7)

axes[-1].set_xlabel("Rater")

fig.legend(
    handles=[
        Patch(facecolor=SIM_COLOR, label="simulated conversations"),
        Patch(facecolor=REAL_COLOR, label="real conversations"),
        Line2D([0], [0], color="grey", linestyle="--", alpha=0.5, label="chance"),
    ],
    loc="upper center", ncol=3, frameon=True, fontsize=10,
    bbox_to_anchor=(0.5, -0.01),
)
fig.suptitle("Per-Rater: % of Conversations Judged as Real (Sequential Protocol)", fontsize=13)
plt.tight_layout(rect=[0, 0.03, 1, 1])

plt.savefig(EXPORT_DIR / "llm_per_rater_sequential.pdf", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Compute CIs for real-only and overall metrics too
def _ci_for_metric(k, n):
    lo, hi = wilson_ci(int(k), int(n))
    return k / n if n else 0, lo, hi

plot_data = []
for _, r in seq_results.iterrows():
    sim_rate, sim_lo, sim_hi = _ci_for_metric(r["tp"], r["n_sim"])
    real_rate, real_lo, real_hi = _ci_for_metric(r["tn"], r["n_real"])
    total_correct = int(r["tp"]) + int(r["tn"])
    total_n = int(r["n_sim"]) + int(r["n_real"])
    overall_rate, overall_lo, overall_hi = _ci_for_metric(total_correct, total_n)
    plot_data.append({
        "label": r["label"],
        "sim_rate": sim_rate, "sim_lo": sim_lo, "sim_hi": sim_hi,
        "real_rate": real_rate, "real_lo": real_lo, "real_hi": real_hi,
        "overall_rate": overall_rate, "overall_lo": overall_lo, "overall_hi": overall_hi,
    })
plot_df = pd.DataFrame(plot_data)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.5), sharey=True)

bar_colors = ["#2E86C1", "#E74C3C"]
y_pos = np.arange(len(plot_df))

metrics = [
    ("sim_rate", "sim_lo", "sim_hi", "Simulation Detection Rate\n(simulated conversations only)"),
    ("real_rate", "real_lo", "real_hi", "Real Recognition Rate\n(real conversations only)"),
    ("overall_rate", "overall_lo", "overall_hi", "Overall Accuracy\n(all conversations)"),
]

for ax, (rate_col, lo_col, hi_col, title) in zip(axes, metrics):
    rates = plot_df[rate_col].values
    los = plot_df[lo_col].values
    his = plot_df[hi_col].values

    ax.barh(y_pos, rates, height=0.5, color=bar_colors, alpha=0.8, zorder=2)
    ax.errorbar(rates, y_pos, xerr=[rates - los, his - rates],
                fmt="none", ecolor="black", capsize=3, linewidth=0.8, zorder=3)

    for i, (r, lo, hi) in enumerate(zip(rates, los, his)):
        ax.text(hi + 0.02, i, f"{r:.0%} [{lo:.0%}, {hi:.0%}]", va="center", fontsize=8)

    ax.axvline(0.50, color="grey", linestyle=":", linewidth=0.8, alpha=0.4)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_df["label"])
    ax.set_xlim(0, 1.0)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.set_title(title, fontsize=10)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

fig.suptitle("Sequential Rater Protocol (95% Wilson CI)", fontsize=13)
plt.tight_layout()

plt.savefig(EXPORT_DIR / "llm_sequential_ci.pdf", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Primary conclusion (per protocol section 16)
for _, r in seq_results.iterrows():
    label = r["label"]
    tp = int(r["tp"])
    n_sim = int(r["n_sim"])
    det = r["det_rate"]
    ub = r["upper_one_sided_95"]
    passes = r["passes_H0"]
    verdict = "met" if passes else "not met"
    below = "below" if passes else "not below"

    print(f"=== {label} ({int(r['n_raters'])} raters) ===")
    print(f"Among actually simulated conversations, {tp} of {n_sim} were classified as simulated.")
    print(f"The estimated simulation detection rate was {tp}/{n_sim} = {det:.1%}.")
    print(f"The one-sided 95% Wilson upper bound was {ub:.1%}.")
    print(f"Because this upper bound was {below} the pre-specified 65% margin, "
          f"the primary criterion was {verdict}.")
    print()

## Learning Effect: Does Detection Improve with Exposure?

If raters learn to detect simulator artefacts over the course of a session, we'd expect detection rate to increase with presentation order. We track this two ways:

- **Per-position** (dots): detection rate at each position across all raters (noisy, small N per point)
- **Cumulative** (line + CI band): detection rate using all judgements from position 1 through k (smooths noise, growing sample)

In [ ]:
# Learning effect: cumulative metrics by presentation order (single plot per judge)
judges_ordered_seq = sorted(SEQUENTIAL_FILES.keys())
positions = range(1, 11)

SIM_COLOR = "#E74C3C"
REAL_COLOR = "#2E86C1"
ACC_COLOR = "#27AE60"

metric_configs = [
    ("simulated", "classified_simulated", "Simulation detection rate", SIM_COLOR, "-o"),
    ("real", "classified_simulated", "Real recognition rate", REAL_COLOR, "-s"),
    (None, "correct", "Overall accuracy", ACC_COLOR, "-^"),
]

fig, axes = plt.subplots(1, len(judges_ordered_seq),
                         figsize=(6 * len(judges_ordered_seq), 4.5), sharey=True)
if len(judges_ordered_seq) == 1:
    axes = [axes]

for col_idx, judge in enumerate(judges_ordered_seq):
    ax = axes[col_idx]

    for gt_filter, count_col, metric_name, color, marker in metric_configs:
        if gt_filter:
            judge_df = seq_df[(seq_df["judge"] == judge) & (seq_df["ground_truth"] == gt_filter)]
        else:
            judge_df = seq_df[seq_df["judge"] == judge]

        cum_rates, cum_los, cum_his = [], [], []
        for k in positions:
            up_to_k = judge_df[judge_df["presentation_order"] <= k]
            n = len(up_to_k)
            if n == 0:
                cum_rates.append(np.nan); cum_los.append(np.nan); cum_his.append(np.nan)
            else:
                if gt_filter == "real":
                    tp = int((~up_to_k["classified_simulated"]).sum())
                elif gt_filter == "simulated":
                    tp = int(up_to_k["classified_simulated"].sum())
                else:
                    tp = int(up_to_k["correct"].sum())
                rate = tp / n
                lo, hi = wilson_ci(tp, n)
                cum_rates.append(rate); cum_los.append(lo); cum_his.append(hi)

        x = list(positions)
        ax.fill_between(x, cum_los, cum_his, alpha=0.10, color=color)
        ax.plot(x, cum_rates, marker, color=color, markersize=5, linewidth=2, zorder=3, label=metric_name)

    ax.axhline(0.50, color="grey", linestyle=":", linewidth=0.8, alpha=0.4, label="Chance")
    ax.set_xticks(list(positions))
    ax.set_xlabel("Presentation order (position)")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlim(0.5, 10.5)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.set_title(judge, fontsize=11)
    if col_idx == 0:
        ax.set_ylabel("Cumulative rate")
        ax.legend(loc="lower right", fontsize=9, frameon=True)

fig.suptitle("Learning Effect: Cumulative Metrics by Presentation Order\n(95% Wilson CI)", fontsize=13)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "llm_learning_effect.pdf", bbox_inches="tight", dpi=150)
plt.show()

## Inter-Rater Agreement (Opus vs GPT)

Each rater index sees the same 10 conversations. We compare how the two LLM judges classified each conversation to measure their agreement.

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Match Opus and GPT ratings by rater_index + conversation_id
opus_df = seq_df[seq_df["judge"] == "claude-opus-4.6"].copy()
gpt_df = seq_df[seq_df["judge"] == "gpt-5.4"].copy()

opus_df["rater_index"] = opus_df["rater_id"].str.rsplit("_", n=1).str[1].astype(int)
gpt_df["rater_index"] = gpt_df["rater_id"].str.rsplit("_", n=1).str[1].astype(int)

merged = opus_df.merge(
    gpt_df,
    on=["rater_index", "conversation_id"],
    suffixes=("_opus", "_gpt"),
)

print(f"Matched {len(merged)} conversation-rater pairs")
print(f"  Unique conversations: {merged['conversation_id'].nunique()}")
print(f"  Unique rater indices: {merged['rater_index'].nunique()}")

# Cohen's kappa on binary classification (simulated vs real)
kappa = cohen_kappa_score(merged["classified_simulated_opus"], merged["classified_simulated_gpt"])

# Agreement breakdown
both_sim = ((merged["classified_simulated_opus"]) & (merged["classified_simulated_gpt"])).sum()
both_real = ((~merged["classified_simulated_opus"]) & (~merged["classified_simulated_gpt"])).sum()
opus_sim_gpt_real = ((merged["classified_simulated_opus"]) & (~merged["classified_simulated_gpt"])).sum()
opus_real_gpt_sim = ((~merged["classified_simulated_opus"]) & (merged["classified_simulated_gpt"])).sum()

agree = both_sim + both_real
total = len(merged)
pct_agree = agree / total

# Correctness breakdown
both_correct = ((merged["correct_opus"]) & (merged["correct_gpt"])).sum()
both_wrong = ((~merged["correct_opus"]) & (~merged["correct_gpt"])).sum()
opus_only = ((merged["correct_opus"]) & (~merged["correct_gpt"])).sum()
gpt_only = ((~merged["correct_opus"]) & (merged["correct_gpt"])).sum()

print(f"\nCohen's κ = {kappa:.3f}")
print(f"Percent agreement: {pct_agree:.1%} ({agree}/{total})")
print(f"\nClassification agreement:")
print(f"  Both → simulated: {both_sim}")
print(f"  Both → real:      {both_real}")
print(f"  Opus → sim, GPT → real: {opus_sim_gpt_real}")
print(f"  Opus → real, GPT → sim: {opus_real_gpt_sim}")
print(f"\nCorrectness breakdown:")
print(f"  Both correct: {both_correct}")
print(f"  Both wrong:   {both_wrong}")
print(f"  Opus only:    {opus_only}")
print(f"  GPT only:     {gpt_only}")

In [ ]:
# Agreement matrix: Opus classification vs GPT classification
fig, ax = plt.subplots(figsize=(5.5, 4.5))

cm_class = np.array([
    [both_real, opus_real_gpt_sim],
    [opus_sim_gpt_real, both_sim],
])
labels_class = ["Real", "Simulated"]
cmap = plt.cm.GnBu

im = ax.imshow(cm_class, cmap=cmap, vmin=0, vmax=cm_class.max(), aspect="equal")
for i in range(2):
    for j in range(2):
        pct = cm_class[i, j] / total * 100
        ax.text(j, i, f"{cm_class[i, j]}\n({pct:.0f}%)",
                ha="center", va="center", fontsize=11,
                color="white" if cm_class[i, j] > cm_class.max() * 0.6 else "black")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(labels_class, fontsize=10)
ax.set_yticklabels(labels_class, fontsize=10)
ax.set_xlabel("GPT-5.4 classification", fontsize=11)
ax.set_ylabel("Opus classification", fontsize=11)
ax.set_title(f"Classification Agreement\n\u03ba = {kappa:.3f}, agreement = {pct_agree:.0%}", fontsize=11)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(left=False, bottom=False)

fig.suptitle("Inter-Rater Agreement: Opus vs GPT (Sequential Protocol)", fontsize=13)
plt.tight_layout()
plt.savefig(EXPORT_DIR / "llm_agreement.pdf", bbox_inches="tight", dpi=150)
plt.show()

## Export Metrics for Report

In [ ]:
EXPORT_DIR = Path("../report/data")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# --- Absolute metrics (train_1, train_2, test) ---
abs_export = []
for _, r in df.iterrows():
    c = r["confusion"]
    abs_export.append({
        "split": r["split"],
        "judge": r["judge"],
        "n_pairs": int(r["n_pairs"]),
        "abs_accuracy": float(r["abs_accuracy"]),
        "abs_acc_sim": float(r["abs_acc_sim"]),
        "abs_acc_real": float(r["abs_acc_real"]),
        "likert_sim": float(r["likert_sim"]),
        "likert_real": float(r["likert_real"]),
        "tp": int(c["tp"]), "tn": int(c["tn"]),
        "fp": int(c["fp"]), "fn": int(c["fn"]),
        "comp_acc": float(r["comp_acc"]) if r["comp_acc"] is not None else None,
    })

with open(EXPORT_DIR / "llm_absolute_metrics.json", "w") as f:
    json.dump(abs_export, f, indent=2)

# --- Sequential protocol metrics ---
seq_export = []
for _, r in seq_results.iterrows():
    seq_export.append({k: (int(v) if isinstance(v, (np.integer,)) else
                           float(v) if isinstance(v, (np.floating,)) else
                           bool(v) if isinstance(v, (np.bool_,)) else v)
                       for k, v in r.to_dict().items()})

with open(EXPORT_DIR / "llm_sequential_metrics.json", "w") as f:
    json.dump(seq_export, f, indent=2)

print(f"Exported to {EXPORT_DIR.resolve()}:")
for p in sorted(EXPORT_DIR.glob("llm_*.json")):
    print(f"  {p.name}")